In [ ]:
#sthaku@ethz.ch, group40

: 

In [ ]:
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import scipy.signal as signal

: 

In [ ]:
# define path to data and sampling frequency
# adjust the path depending on if you run this script locally or on Kaggle
DATA_PATH = '/kaggle/input/competitions/ppg-hr-exercise-2/mhealth26_ex2.npy'  # for Kaggle
# DATA_PATH = './mhealth26_ex2.npy'  # when running locally
FS = 128

In [ ]:
# data loader class to access the data from the .npy file
class MHealthDataLoader:
    def __init__(self, path):
        self.data = np.load(path, allow_pickle=True).item()
        self.n_windows = len(self.data['ppg'])
        self.splits = self.data['split']

    def get_window(self, idx):
        # get ppg signal
        ppg = self.data['ppg'][idx]
        # transforming list accelerometer axis into one array: shape is now (3, ...) where ... is the time axis
        acc = np.array(self.data['imu'][idx], dtype=np.int32)
        # calculating the accelerometer magnitude as sqrt(x^2 + y^2 + z^2)
        mag = np.square(acc)
        mag = np.sqrt(np.sum(mag, axis=0))
        # adding the magnitude to the 2D array: the shape is now (4, ...) where ... is the time axis
        acc = np.vstack([acc, mag[:,None].reshape(1, acc.shape[1])])

        # does this window belong to test or train split?
        split = self.splits[idx]
        # get hr labels: if it is a test window, this list is empty, otherwise there is one hr label for each window of 30sec*128Hz samples of acc and ppg
        hr = np.array(self.data['hr'][idx])
        
        return ppg, acc, hr, split

    def __call__(self):
        return [self.get_window(i) for i in range(self.n_windows)]

In [ ]:
# example function to plot any signal with time on the x-axis
def plot_signal(signal, title, ylabel, sampling_rate=128):
    x = np.linspace(0, len(signal) / sampling_rate, len(signal))
    t = pd.to_datetime(x, unit='s')

    fig, ax = plt.subplots()
    ax.plot(t, signal)
    ax.set_title(title)
    ax.set_xlabel('Time [min:sec]')
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%M:%S'))
    plt.show()

In [ ]:
# define the data loader and get the data
# the file mhealth26_ex2.npy consists of 32 windows, with 20 train and 12 test windows
# each window consists of 4 elements
# w[0] is the ppg signal, w[1][0:2] are the three axis of the accelerometer signal, w[1][3] is the magnitude of the accelerometer signal,
# w[2] is the heart rate label (only available for the train windows!), and w[3] is the split ('train' and 'test')
get_data = MHealthDataLoader(DATA_PATH)
windows = get_data()

In [ ]:
# plot the first 30 seconds of the unfiltered ppg signal of the very first window
plot_signal(windows[21][0][0:30*FS], 'PPG Signal', 'PPG Signal')

In [ ]:
# function to predict HR using a window of 30 seconds
def predict_hr(window):
    ppg, acc, hr, split = window
    n_preds = len(ppg) // FS // 30

    preds = np.zeros(n_preds, dtype=np.float32)

    # ToDo: implement your HR prediction algorithm here
    # this is a dummy example that predicts random values
    for i in range(n_preds):
        
        preds[i] = np.random.randint(40, 180)

    return preds

In [ ]:
# function to print the mean and median absolute error between your predicted HR and the HR labels
# with this function, you can evaluate the resulting score that you would obtain on the train set
# with your predicted HR values
def print_score(pred_hr, label_hr):
    err = np.abs(np.asarray(pred_hr) - np.asarray(label_hr))
    print("Mean error: {:4.3f}, Median error {:4.3f}".format(np.mean(err), np.median(err)))
    print("Resulting score {:4.3f}".format(0.5 * np.mean(err) + 0.5 * np.median(err)))

In [ ]:
# create the submission file called "submission.csv"
# download the file and submit it to the kaggle competition to obtain a score on the leaderboard
idxs = []
preds = []

# loop through all windows
# important: loop through all windows as below, add the predictions in the same order and keep the naming of window{i}_{j}
# we will only evaluate your predictions for the test set (see description) but the predictions for the training set have to be in the 
# submission.csv file as well to ensure correct matching of your results with solution file
for i, w in enumerate(windows):
    n_preds = len(w[0]) // FS // 30
    # index needed for kaggle submission file
    idxs.extend([f"window{i}_{j}" for j in range(n_preds)])
    # your predictions based on the above function
    preds.extend(predict_hr(w))

# on the right side in tab "Output", you should now see a file called "submission.csv" after pressing the refresh button
# download the file and submit it to this competition to obtain a score on the leaderboard
df = pd.DataFrame({'Id': idxs, 'Predicted': preds})
# df.to_csv('/kaggle/working/submission.csv', index=False)  # for Kaggle
df.to_csv('./submission.csv', index=False)  # when running locally

In [ ]:
# example on how to use the print_score function with randomly generated HR values as the predictions and labels
pred_hr_random = list(np.random.randint(40, 180, len(preds)))
label_hr_random = list(np.random.randint(40, 180, len(preds)))
print_score(pred_hr_random, label_hr_random)